# StreamCut — Colabで話題別の字幕付き下書きを作る

初回はGPUランタイム1つ・`role=single`で短い動画を試してください。
複数台では同じノートブックを各ランタイムで開き、同じproject_nameとコード版を指定します。
GPU4＋CPU4は、coordinator 1台・GPU rank 0〜3・render rank 1〜3です。

自動ランタイム確保は含みません。DriveとColabのアカウントを確認し、APIキーはColab Secretsへ保存します。


In [ ]:
#@title 1. 設定
role = "single" #@param ["single", "coordinator", "gpu", "render"]
rank = 0 #@param {type:"integer"}
gpu_workers = 1 #@param {type:"integer"}
render_workers = 1 #@param {type:"integer"}
project_name = "job-001" #@param {type:"string"}
source = "/content/drive/MyDrive/Inputs/source.mp4" #@param {type:"string"}
llm_model = "" #@param {type:"string"}
llm_base_url = "https://api.openai.com/v1" #@param {type:"string"}
api_key_secret = "STREAMCUT_API_KEY" #@param {type:"string"}
clips = 3 #@param {type:"integer"}
batch_size = 8 #@param {type:"integer"}
burn_in = True #@param {type:"boolean"}
silence_cut = True #@param {type:"boolean"}
resume = False #@param {type:"boolean"}
release_runtime = True #@param {type:"boolean"}
revision = "v0.1.1" #@param {type:"string"}

import re
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,80}", project_name):
    raise ValueError("project_nameは英数字・ハイフン・アンダースコアで指定してください")
if not re.fullmatch(r"[A-Za-z0-9._/-]+", revision):
    raise ValueError("コードの版が不正です")
if role in {"single", "coordinator"} and not llm_model.strip():
    raise ValueError("利用するllm_modelを指定してください。既定モデルはありません")
if not 1 <= gpu_workers <= 4 or not 1 <= render_workers <= 4:
    raise ValueError("ワーカー数は1〜4です")
if role == "single":
    gpu_workers = render_workers = 1


In [ ]:
#@title 2. Driveと実行環境の準備
import json
import os
from pathlib import Path
import subprocess
import sys
import time
from google.colab import drive

drive.mount("/content/drive")
project = Path("/content/drive/MyDrive/StreamCut") / project_name
setup_started = time.monotonic()
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "fontconfig", "fonts-noto-cjk"], check=True)
checkout = Path("/content/streamcut-checkout")
if not checkout.exists():
    subprocess.run(["git", "clone", "--no-checkout", "https://github.com/nyu1791-collab/StreamCut.git", str(checkout)], check=True)
subprocess.run(["git", "-C", str(checkout), "fetch", "origin", revision], check=True)
subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", "FETCH_HEAD"], check=True)
import runpy
extras = "gpu,fetch" if role in {"single", "gpu"} else "vad,fetch"
bootstrap = runpy.run_path(str(checkout / "tools/colab_bootstrap.py"))["bootstrap"]
python = bootstrap(checkout, extras)
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"
# Expose installed CUDA libraries to CTranslate2 without importing torch into the notebook kernel.
site_dirs = json.loads(subprocess.check_output([python, "-c", "import json,site;print(json.dumps(site.getsitepackages()))"], text=True))
lib_dirs = [str(Path(base) / "nvidia" / name / "lib") for base in site_dirs
            for name in ("cublas", "cudnn") if (Path(base) / "nvidia" / name / "lib").exists()]
if lib_dirs:
    env["LD_LIBRARY_PATH"] = ":".join(lib_dirs + [env.get("LD_LIBRARY_PATH", "")])
doctor = json.loads(subprocess.check_output([python, "-m", "streamcut.cli", "doctor"], env=env, text=True))
print(json.dumps(doctor, ensure_ascii=False, indent=2))
if role in {"single", "gpu"} and doctor["cuda_devices"] < 1:
    raise RuntimeError("GPUランタイムを選択してください。CPU推論へ自動切替しません")
print("Colab setup seconds:", round(time.monotonic() - setup_started, 2))


In [ ]:
#@title 3. 実行（工程間の手渡しは不要）
from google.colab import runtime, userdata

if role in {"single", "coordinator"}:
    env["STREAMCUT_API_KEY"] = userdata.get(api_key_secret)
    settings = {"llm_base_url": llm_base_url, "llm_model": llm_model,
                "clips": clips, "batch_size": batch_size,
                "burn_in": burn_in, "silence_cut": silence_cut,
                "gpu_workers": gpu_workers, "render_workers": render_workers}
    settings_path = Path("/content/streamcut-settings.json")
    settings_path.write_text(json.dumps(settings))
    args = [python, "-m", "streamcut.cli", "run", source,
            "--project", str(project), "--config", str(settings_path)]
    if role == "coordinator":
        args.append("--distributed")
    if resume:
        args.append("--resume")
else:
    args = [python, "-m", "streamcut.cli", "worker", str(project),
            "--role", role, "--rank", str(rank)]
try:
    subprocess.run(args, env=env, check=True)
    print("成果物:", str(project / "clips"))
finally:
    env.pop("STREAMCUT_API_KEY", None)
    if release_runtime:
        try:
            runtime.unassign()
        except Exception as error:
            print("ランタイム解放を確認できません。手動で停止してください:", type(error).__name__)


## 確認するもの

`clips/01/video.mp4`、`captions.srt`、`render.json` を確認してください。
自動検査の完了は、日本語・話題・画面内容を人が確認した意味ではありません。

失敗した場合は全ワーカーを停止し、原因を修正して同じ設定とrankで再開します。
coordinator側でresumeを有効にすると成功済みタスクを再利用します。
強制終了後のロックやDriveの注意点はリポジトリのdocs/COLAB.mdを参照してください。
